# Image Super-Resolution with ANNs — DIV2K Tutorial

Full pipeline for 2× super-resolution on the DIV2K dataset:

1. **Data preparation** — extract 64×64 HR / 32×32 LR patch pairs from DIV2K
2. **Baseline** — bicubic & bilinear upsampling + PSNR/SSIM
3. **ESPCN model** — Conv → Conv → Conv → PixelShuffle(2)
4. **Training** — L1 (MAE) loss, Adam optimizer, 20–50 epochs
5. **Evaluation** — PSNR / SSIM comparison
6. **Visual comparison** — side-by-side: LR-upsampled · ESPCN · HR ground truth

In [ ]:
import sys
import os

sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from torchvision import transforms
from pathlib import Path

print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {"cuda" if torch.cuda.is_available() else "cpu"}')

HR_DIR   = '../Data/DIV2K_valid_HR'
LR_DIR   = '../Data/DIV2K_valid_LR_bicubic/X2'
OUT_DIR  = '../Data'
SCALE    = 2
PATCH_SIZE = 64
STRIDE     = 64
NUM_TRAIN_IMAGES = 80
CHECKPOINT = '../outputs/checkpoints/espcn_best.pth'

## 1. Data Preparation

We use **DIV2K validation HR images** (0801–0900) as the source.  
`prepare_data.py` extracts aligned **64×64 HR** and **32×32 LR** patches.  
The matching LR images already exist in `Data/DIV2K_valid_LR_bicubic/X2/` (named `0801x2.png`, etc.).

In [ ]:
from prepare_data import extract_patches

hr_patches, lr_patches = extract_patches(
    hr_dir=HR_DIR,
    lr_dir=LR_DIR,
    patch_size=PATCH_SIZE,
    scale_factor=SCALE,
    stride=STRIDE,
    num_images=NUM_TRAIN_IMAGES,
)

os.makedirs(OUT_DIR, exist_ok=True)
np.save(os.path.join(OUT_DIR, 'hr_patches.npy'), hr_patches)
np.save(os.path.join(OUT_DIR, 'lr_patches.npy'), lr_patches)
print(f'Saved {len(hr_patches)} patch pairs to {OUT_DIR}/')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    axes[0, i].imshow(hr_patches[i])
    axes[0, i].set_title(f'HR patch {i}  (64×64)')
    axes[0, i].axis('off')
    axes[1, i].imshow(lr_patches[i])
    axes[1, i].set_title(f'LR patch {i}  (32×32)')
    axes[1, i].axis('off')
plt.suptitle('Sample HR / LR patch pairs  (2× scale)', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Baseline Evaluation

Bicubic / bilinear upsampling of the **last 20 validation images** (held-out test set).

In [ ]:
from baseline import evaluate_baseline

bicubic_psnr, bicubic_ssim = evaluate_baseline(
    hr_dir=HR_DIR, lr_dir=LR_DIR,
    scale_factor=SCALE, method='bicubic',
    output_dir='../outputs/comparison_images',
)

bilinear_psnr, bilinear_ssim = evaluate_baseline(
    hr_dir=HR_DIR, lr_dir=LR_DIR,
    scale_factor=SCALE, method='bilinear',
)

## 3. Model Architecture — ESPCN

**ESPCN** (Efficient Sub-Pixel CNN) operates at native LR resolution and uses a learned PixelShuffle to upsample:

```
Input (32×32 LR)
  └─ Conv2d(3→64,  5×5) + ReLU
  └─ Conv2d(64→32, 3×3) + ReLU
  └─ Conv2d(32→3·4, 3×3)      ← 3 × 2² = 12 output channels
  └─ PixelShuffle(2)           ← rearrange to 64×64
Output (64×64 SR)
```

In [ ]:
from model import build_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

espcn = build_model('espcn', scale_factor=SCALE).to(device)
print('=== ESPCN (x2) ===')
print(espcn)
print(f'Parameters: {sum(p.numel() for p in espcn.parameters()):,}')

x_lr = torch.randn(1, 3, 32, 32).to(device)
x_sr = espcn(x_lr)
print(f'\nForward pass: {tuple(x_lr.shape)}  →  {tuple(x_sr.shape)}')

## 4. Training

- **Loss**: L1 (MAE) — produces sharper results than MSE
- **Optimizer**: Adam (lr=1e-4)
- **Epochs**: 20–50 depending on compute
- **Patches**: 64×64 HR ↔ 32×32 LR from DIV2K

In [ ]:
import argparse
from train import train

args = argparse.Namespace(
    arch='espcn',
    hr_dir=HR_DIR,
    lr_dir=LR_DIR,
    scale=SCALE,
    epochs=50,
    batch_size=16,
    lr=1e-4,
    lr_step=15,
    checkpoint_dir='../outputs/checkpoints',
)

train(args)

## 5. Quantitative Evaluation (PSNR / SSIM)

Compare the trained ESPCN model against the bicubic baseline on the full validation set.

In [ ]:
from evaluate import evaluate_model

if Path(CHECKPOINT).exists():
    model_psnr, model_ssim = evaluate_model(
        checkpoint_path=CHECKPOINT,
        hr_dir=HR_DIR,
        lr_dir=LR_DIR,
        arch='espcn',
        scale=SCALE,
        output_dir='../outputs/predictions',
    )
else:
    print(f'Checkpoint not found: {CHECKPOINT}')
    print('Run the training cell first.')
    model_psnr, model_ssim = [], []

In [ ]:
methods   = ['Bilinear', 'Bicubic']
psnr_vals = [np.mean(bilinear_psnr) if bilinear_psnr else 0,
             np.mean(bicubic_psnr)  if bicubic_psnr  else 0]
ssim_vals = [np.mean(bilinear_ssim) if bilinear_ssim else 0,
             np.mean(bicubic_ssim)  if bicubic_ssim  else 0]

if model_psnr:
    methods.append('ESPCN')
    psnr_vals.append(np.mean(model_psnr))
    ssim_vals.append(np.mean(model_ssim))

colors = ['#4C72B0', '#DD8452', '#55A868']
x = np.arange(len(methods))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

bars1 = ax1.bar(x, psnr_vals, color=colors[:len(methods)])
ax1.set_xticks(x); ax1.set_xticklabels(methods)
ax1.set_ylabel('PSNR (dB)'); ax1.set_title('Mean PSNR  (higher is better)')
for bar, v in zip(bars1, psnr_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, v + 0.1, f'{v:.2f}', ha='center', fontsize=10)

bars2 = ax2.bar(x, ssim_vals, color=colors[:len(methods)])
ax2.set_xticks(x); ax2.set_xticklabels(methods)
ax2.set_ylabel('SSIM'); ax2.set_title('Mean SSIM  (higher is better)')
for bar, v in zip(bars2, ssim_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.4f}', ha='center', fontsize=10)

plt.suptitle(f'Super-Resolution Comparison — DIV2K x{SCALE}', fontsize=13)
plt.tight_layout()
plt.show()

print('\n=== Summary ===')
for m, p, s in zip(methods, psnr_vals, ssim_vals):
    print(f'  {m:<10}  PSNR={p:.2f} dB   SSIM={s:.4f}')

## 6. Visual Comparison

Side-by-side: **LR (bicubic upsampled)** · **ESPCN output** · **HR ground truth**

In [ ]:
from baseline import bicubic_upsample

hr_files = sorted(Path(HR_DIR).glob('*.png'))
lr_files = sorted(Path(LR_DIR).glob('*.png'))

if not hr_files:
    print('No HR images found.')
else:
    sample_hr = Image.open(hr_files[0]).convert('RGB')
    sample_lr = Image.open(lr_files[0]).convert('RGB')
    bicubic_img = bicubic_upsample(sample_lr, SCALE)

    sr_img = None
    if Path(CHECKPOINT).exists():
        model = build_model('espcn', scale_factor=SCALE).to(device)
        ckpt  = torch.load(CHECKPOINT, map_location=device)
        model.load_state_dict(ckpt.get('model_state', ckpt))
        model.eval()
        lr_t = transforms.ToTensor()(sample_lr).unsqueeze(0).to(device)
        with torch.no_grad():
            sr_t = model(lr_t).clamp(0, 1)
        sr_img = transforms.ToPILImage()(sr_t.squeeze(0).cpu())

    CROP_W, CROP_H = 256, 256
    def crop(img):
        return img.crop((0, 0, min(img.width, CROP_W), min(img.height, CROP_H)))

    cols = [
        ('LR → Bicubic',        bicubic_img),
        ('HR (ground truth)',   sample_hr),
    ]
    if sr_img:
        cols.insert(1, ('ESPCN (ours)', sr_img))

    fig, axes = plt.subplots(1, len(cols), figsize=(5 * len(cols), 5))
    for ax, (title, img) in zip(axes, cols):
        ax.imshow(crop(img))
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.axis('off')

    plt.suptitle(f'{hr_files[0].name}  —  2× Super-Resolution', fontsize=13)
    plt.tight_layout()
    plt.show()

    if sr_img:
        from evaluate import compute_psnr, compute_ssim
        sr_arr  = np.array(sr_img,     dtype=np.float32) / 255.0
        bic_arr = np.array(bicubic_img, dtype=np.float32) / 255.0
        hr_arr  = np.array(sample_hr,  dtype=np.float32) / 255.0
        min_h = min(sr_arr.shape[0], hr_arr.shape[0])
        min_w = min(sr_arr.shape[1], hr_arr.shape[1])
        sr_arr  = sr_arr[:min_h, :min_w]
        bic_arr = bic_arr[:min_h, :min_w]
        hr_arr  = hr_arr[:min_h, :min_w]
        print(f"Bicubic  PSNR={compute_psnr(hr_arr, bic_arr):.2f} dB   SSIM={compute_ssim(hr_arr, bic_arr):.4f}")
        print(f"ESPCN    PSNR={compute_psnr(hr_arr, sr_arr):.2f} dB   SSIM={compute_ssim(hr_arr, sr_arr):.4f}")

## 7. Comparison Story

### Why does the CNN outperform bicubic?

**Bicubic upsampling** is a fixed mathematical filter — it smoothly interpolates pixel values but has no knowledge of natural image structure. It blurs edges and produces ringing artefacts, especially at fine texture regions.

**ESPCN** learns to *reconstruct* high-frequency detail from the training data:
- The three convolutional layers build a feature hierarchy of local patterns
- The PixelShuffle (sub-pixel convolution) layer recombines those features into a sharp, high-resolution output
- Trained with **L1 loss**, it is penalised equally for every pixel error — this pushes the network towards sharper edges rather than the over-smooth "average" that MSE tends to produce

### What the numbers tell us

| Method   | PSNR (dB) ↑ | SSIM ↑ |
|----------|------------|--------|
| Bilinear | ~          | ~      |
| Bicubic  | ~          | ~      |
| ESPCN    | ~          | ~      |

*(Fill in after running Cells 2 and 5)*

A **+0.5 dB PSNR improvement** over bicubic corresponds to roughly a 12 % reduction in pixel-level error. SSIM gains reflect better preservation of local contrast and structure — the kind of detail that matters perceptually (textures, edges, fine lines).

### Key takeaways
1. **CNN learns texture priors** that fixed filters cannot capture
2. **L1 loss → sharper outputs** than L2/MSE for the same architecture
3. **PixelShuffle is efficient** — the whole upsampling computation runs at LR resolution, making ESPCN faster than methods that upsample first